# Creatix — Evaluation Notebook

This notebook runs the evaluation described in **Section 9** of the capstone documentation against the
**live deployed Creatix backend** (`https://creatix-backend-y8oh.onrender.com`). It measures:

1. Task-routing accuracy (Planner Agent)
2. Reviewer approval rate & average score
3. Retrieval relevance (Repository Q&A) — requires one manual judgement step
4. End-to-end response latency per task type

Run all cells top to bottom. The final cell prints a results table matching Section 9.2 of the report.

> Note: the backend is hosted on Render's free tier, so the **first call may take 20–50s** while the
> service wakes up from a cold start. This is expected — the warm-up cell below absorbs that delay so it
> doesn't distort the latency measurements.


In [ ]:
import requests
import time
import statistics
import pandas as pd

BACKEND_URL = "https://creatix-backend-y8oh.onrender.com/generate"
TEST_REPO = "https://github.com/rishita1-dev/Creatix"

def call_creatix(task, prompt="", repo_url=None, timeout=180):
    """Call the Creatix /generate endpoint and return (response_json, latency_seconds)."""
    payload = {"task": task, "prompt": prompt}
    if repo_url:
        payload["repo_url"] = repo_url
    start = time.time()
    resp = requests.post(BACKEND_URL, json=payload, timeout=timeout)
    latency = time.time() - start
    resp.raise_for_status()
    return resp.json(), latency


## 0. Warm-up call (absorbs Render cold-start delay, not counted in latency results)

In [ ]:
_ = call_creatix("Generate Code", "print('warming up the backend')")
print("Backend is warm.")


## 1. Task-Routing Accuracy (Planner Agent)

Each prompt below is sent with `task="Auto"`. We compare the Planner's `selected_task` against the
`expected_task` we intended.

In [ ]:
routing_tests = [
    {"prompt": "Write a Python function that checks if a string is a palindrome.",
     "repo_url": None, "expected_task": "Generate Code"},
    {"prompt": "Explain what this code does: def f(n): return n*f(n-1) if n>1 else 1",
     "repo_url": None, "expected_task": "Explain Code"},
    {"prompt": "This function throws IndexError on an empty list, why? def first(l): return l[0]",
     "repo_url": None, "expected_task": "Debug Code"},
    {"prompt": "Review this repo's code quality.",
     "repo_url": TEST_REPO, "expected_task": "Review GitHub Repository"},
    {"prompt": "Where is the FAISS index built in this repository?",
     "repo_url": TEST_REPO, "expected_task": "Repository Q&A (RAG)"},
    {"prompt": "My code isn't working, can you fix it and tell me why it broke: for i in range(10) print(i)",
     "repo_url": None, "expected_task": "Debug Code"},
    {"prompt": "Generate a REST API endpoint in FastAPI for user login.",
     "repo_url": None, "expected_task": "Generate Code"},
    {"prompt": "What's wrong with this SQL query and how do I fix it: SELECT * FROM users WHERE name = John",
     "repo_url": None, "expected_task": "Debug Code"},
]

routing_results = []
for t in routing_tests:
    data, latency = call_creatix("Auto", prompt=t["prompt"], repo_url=t["repo_url"])
    routing_results.append({
        "prompt": t["prompt"][:60] + ("..." if len(t["prompt"]) > 60 else ""),
        "expected_task": t["expected_task"],
        "selected_task": data.get("selected_task"),
        "correct": data.get("selected_task") == t["expected_task"],
        "review_score": data.get("review", {}).get("score") if isinstance(data.get("review"), dict) else None,
        "revised": data.get("revised"),
        "latency_s": round(latency, 2),
    })

routing_df = pd.DataFrame(routing_results)
routing_df


In [ ]:
routing_accuracy = routing_df["correct"].mean() * 100
print(f"Task-routing accuracy: {routing_accuracy:.1f}%  ({routing_df['correct'].sum()}/{len(routing_df)} correct)")


## 2. Reviewer Approval Rate & Average Score

Reuses the Reviewer Agent scores captured in the routing test responses above (adjust the score threshold
if your Reviewer Agent uses a different bar than 7/10).

In [ ]:
scored = routing_df.dropna(subset=["review_score"])
approval_rate = (scored["review_score"] >= 7).mean() * 100 if len(scored) else float("nan")
avg_score = scored["review_score"].mean() if len(scored) else float("nan")

print(f"Reviewer approval rate (score >= 7): {approval_rate:.1f}%")
print(f"Average reviewer score: {avg_score:.2f}/10")
print(f"Responses revised: {routing_df['revised'].sum()} / {len(routing_df)}")


## 3. Retrieval Relevance (Repository Q&A)

We ask five factual questions about the Creatix repo itself and print each answer. **Manually read each
answer** and set `True`/`False` in the `grounded` list below based on whether the answer correctly reflects
the actual codebase (this is a human-judgement step, same as described in Section 9.1).

In [ ]:
rag_questions = [
    "Where is the FAISS index built in this repository?",
    "What embedding model does this repository use?",
    "How does the Reviewer Agent decide to trigger a revision?",
    "What chunk size and overlap are used for splitting files?",
    "Which library is used to talk to the GitHub API?",
]

rag_results = []
for q in rag_questions:
    data, latency = call_creatix("Repository Q&A (RAG)", prompt=q, repo_url=TEST_REPO)
    rag_results.append({"question": q, "answer": data.get("response"), "latency_s": round(latency, 2)})

for r in rag_results:
    print("Q:", r["question"])
    print("A:", r["answer"])
    print("-" * 80)


In [ ]:
# After reading the answers above, fill in True/False for each question in the SAME order as rag_questions.
# Example: grounded = [True, True, False, True, True]
grounded = [None, None, None, None, None]  # <-- fill this in manually before running the next cell

assert all(g is not None for g in grounded), "Fill in the `grounded` list above with True/False for each answer."

for r, g in zip(rag_results, grounded):
    r["grounded"] = g

rag_df = pd.DataFrame(rag_results)
retrieval_relevance = rag_df["grounded"].mean() * 100
print(f"Retrieval relevance: {retrieval_relevance:.1f}%  ({rag_df['grounded'].sum()}/{len(rag_df)} correctly grounded)")
rag_df


## 4. End-to-End Response Latency

In [ ]:
latency_by_task = routing_df.groupby("expected_task")["latency_s"].mean().round(2)
rag_latency = rag_df["latency_s"].mean()

print("Average latency by task (Auto-routed prompts):")
print(latency_by_task)
print()
print(f"Average Repository Q&A latency: {rag_latency:.2f}s "
      f"(first call includes clone+index build, later calls reuse the cached repo)")


## 5. Summary — copy these numbers into Section 9.2 of the report

In [ ]:
summary = pd.DataFrame([
    {"Metric": "Task-routing accuracy", "Your System": f"{routing_accuracy:.1f}%"},
    {"Metric": "Reviewer approval rate", "Your System": f"{approval_rate:.1f}%"},
    {"Metric": "Average reviewer score", "Your System": f"{avg_score:.2f}/10"},
    {"Metric": "Retrieval relevance", "Your System": f"{retrieval_relevance:.1f}%"},
    {"Metric": "Avg latency — Generate/Explain/Debug", "Your System": f"{latency_by_task.filter(like='Code').mean():.2f}s"},
    {"Metric": "Avg latency — Repository Q&A", "Your System": f"{rag_latency:.2f}s"},
])
summary
